# Colab 12 — Train on artificial AA, transfer to real CATH AA + SS

## What this notebook tests

Iters 11/11b/11c diagnosed that the convolutional Siamese encoder learns **shared local pattern overlap** as its similarity feature, not alignment-style edit-distance computation. This works on perturbation pairs (which share long contiguous substrings) and fails on unrelated natural CATH-S20 pairs (which by construction share nothing).

We now ask three sequential transfer questions, training only on pure synthetic data and testing across a transfer ladder:

| Step | Train on | Eval on | Question |
|---|---|---|---|
| 1 | random AA perturbations | random AA perturbations | does the architecture approximate normLev? (sanity — colab10's recipe at MAX_LEN=200) |
| 2 | random AA perturbations | **real CATH AA perturbations** (test30 seeds) | does the function generalize from uniform to biological AA? |
| 3 | random AA perturbations | **real CATH SS_seq perturbations** (test30 SS seeds) | does the function transfer cross-representation when we shrink to a 3-letter alphabet? |

**No natural-pair tests** in this notebook. We've already established that the architecture cannot approximate Lev on unrelated natural pairs — that's an architectural limitation, not a training-data issue. Here we focus on what the architecture *can* do (perturbation-style Lev approximation) and ask how far it transfers.

After training, encoder weights are not updated during evaluation (`model.eval()` plus no backward pass). That's effectively *freezing* the encoder for the cross-rep test.

## Architecture

Same as colab11/11b/11c (small encoder, ~1.6M params). MAX_LEN=200, vocab 21 (20 AA + PAD). For SS_seq, we use the same vocab — H/L/S already exist as letters in the AA alphabet (Histidine, Leucine, Serine). The encoder will see SS_seq strings as 'AA' sequences that happen to use only 3 of the 20 letters; whether its features still produce useful similarity predictions is the cross-rep question.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'                    # 3-letter secondary structure (H, L, S already exist as AA letters)
VOCAB_SIZE = len(AA_ALPHABET) + 1      # 20 AA + PAD
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}   # H, L, S map to existing AA indices

MIN_LEN = 50
MAX_LEN_SEQ = 200      # max length of generated/loaded sequences before perturbation
MAX_LEN = 200          # padding length

N_TRAIN = 30000
N_HELD = 5000
NUM_EPOCHS = 30
BATCH_SIZE = 128

# H, L, S → AA indices
for c in SS_ALPHABET:
    print(f"'{c}' → AA index {CHAR_TO_IDX[c]}")

## 3. Helpers — Levenshtein, perturb, encode

`perturb` takes an alphabet parameter so we can perturb AA strings with the 20-letter set and SS strings with `HLS` only.

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - levenshtein(a, b) / L

def is_standard_aa(seq):
    return all(c in AA_SET for c in seq)

def is_standard_ss(seq):
    return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, max_len=MAX_LEN):
    s = list(seq)
    abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0:
            op = 'ins'
        elif len(s) >= max_len:
            op = rng.choice(['sub', 'del'])
        else:
            op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in abc if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def random_aa_seq(min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

## 4. Build training pairs — random AA perturbations only

In [ ]:
def make_perturbation_pairs(seed_fn, alphabet, n_pairs):
    """seed_fn: callable that returns one seed sequence each call."""
    pairs = []
    attempts = 0
    max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = seed_fn()
        if seed is None or len(seed) < MIN_LEN: continue
        k = int(rng.integers(1, max(2, int(0.8 * len(seed)))))
        other = perturb(seed, k, alphabet)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

print('Building training pairs (random AA perturbations)…')
train_pairs = make_perturbation_pairs(
    seed_fn=random_aa_seq, alphabet=AA_ALPHABET, n_pairs=N_TRAIN)
print(f'  got {len(train_pairs)} training pairs')

labels = np.array([p[2] for p in train_pairs])
plt.figure(figsize=(8, 3))
plt.hist(labels, bins=40, edgecolor='k', alpha=0.75)
plt.xlabel('Label (normLev similarity)'); plt.ylabel('Count')
plt.title(f'Training-pair label distribution ({len(train_pairs)} random-AA perturbation pairs)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Dataset / Architecture

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b, label = self.pairs[i]
        return encode_pad(a), encode_pad(b), torch.tensor(label, dtype=torch.float32)

train_dl = DataLoader(PairDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True)

class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 max_len=MAX_LEN, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2 * max_len, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x * mask.unsqueeze(1)
        x = x.flatten(1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)

class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__(); self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        ea, eb = self.encoder(a), self.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0
    def encode(self, x): return self.encoder(x)

model = SiameseNetwork().to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
model.train()
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0; nb = 0
    for a, b, label in train_dl:
        a = a.to(device); b = b.to(device); label = label.to(device)
        pred = model(a, b)
        loss = F.mse_loss(pred, label)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} — MSE: {avg:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Training loss — random-AA perturbation pairs')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final MSE: {losses[-1]:.5f}')

## 7. Build held-out eval sets — three transfer levels

Encoder is now effectively frozen — `model.eval()` and no backward pass during evaluation.

In [ ]:
# Load test30 proteins for the real-CATH evaluations
test_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'),
                     compression='gzip').dropna(subset=['aa_seq', 'ss_seq'])
test_df['aa_seq'] = test_df['aa_seq'].astype(str)
test_df['ss_seq'] = test_df['ss_seq'].astype(str)
test_df = test_df[test_df['aa_seq'].str.len().between(MIN_LEN, MAX_LEN_SEQ)]
test_df = test_df[test_df['aa_seq'].apply(is_standard_aa)]
test_df = test_df[test_df['ss_seq'].apply(is_standard_ss)]
test_df = test_df[test_df['aa_seq'].str.len() == test_df['ss_seq'].str.len()]
print(f'test30 proteins kept after filter: {len(test_df)}')

test_aa_seqs = test_df['aa_seq'].tolist()
test_ss_seqs = test_df['ss_seq'].tolist()

# Held-out 1: random AA perturbation pairs (in-domain)
print('\nBuilding held-out 1: random-AA perturbations (in-domain)…')
held_random = make_perturbation_pairs(
    seed_fn=random_aa_seq, alphabet=AA_ALPHABET, n_pairs=N_HELD)
print(f'  got {len(held_random)}')

# Held-out 2: real CATH AA perturbation pairs (distribution shift)
print('Building held-out 2: real CATH AA perturbations (distribution shift)…')
def cath_aa_seed():
    return test_aa_seqs[int(rng.integers(0, len(test_aa_seqs)))]
held_cath_aa = make_perturbation_pairs(
    seed_fn=cath_aa_seed, alphabet=AA_ALPHABET, n_pairs=N_HELD)
print(f'  got {len(held_cath_aa)}')

# Held-out 3: real CATH SS perturbation pairs (cross-rep)
print('Building held-out 3: real CATH SS perturbations (cross-rep)…')
def cath_ss_seed():
    return test_ss_seqs[int(rng.integers(0, len(test_ss_seqs)))]
held_cath_ss = make_perturbation_pairs(
    seed_fn=cath_ss_seed, alphabet=SS_ALPHABET, n_pairs=N_HELD)
print(f'  got {len(held_cath_ss)}')

## 8. V1 — three transfer levels side by side

In [ ]:
def predict_pairs(pairs, batch_size=512):
    model.eval()
    true_v, pred_v = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            pred = model(a, b).cpu().numpy()
            true_v.extend([p[2] for p in batch])
            pred_v.extend(pred.tolist())
    return np.array(true_v), np.array(pred_v)

def report(name, true_v, pred_v):
    if len(true_v) < 10:
        print(f'{name}: n={len(true_v)} too few'); return None
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    mask = true_v > 0.5
    sr_high = spearmanr(true_v[mask], pred_v[mask])[0] if mask.sum() > 10 else float('nan')
    print(f'{name:>26}: n={len(true_v):>5}  Pearson={pr:+.3f}  Spearman={sr:+.3f}  Spearman(>0.5)={sr_high:+.3f}')
    return pr, sr

true_r, pred_r = predict_pairs(held_random)
true_a, pred_a = predict_pairs(held_cath_aa)
true_s, pred_s = predict_pairs(held_cath_ss)

print('=== V1 transfer ladder ===')
report('1. random AA  (in-domain)',     true_r, pred_r)
report('2. CATH AA    (dist. shift)',   true_a, pred_a)
report('3. CATH SS    (cross-rep)',     true_s, pred_s)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (true_v, pred_v, title) in zip(axes, [
    (true_r, pred_r, '1. Random AA (in-domain)'),
    (true_a, pred_a, '2. CATH AA (distribution shift)'),
    (true_s, pred_s, '3. CATH SS (cross-representation)'),
]):
    if len(true_v) < 10:
        ax.set_title(f'{title}\n(too few)'); continue
    ax.scatter(true_v, pred_v, alpha=0.15, s=6)
    ax.plot([0, 1], [0, 1], 'r-', linewidth=2, label='y = x')
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    ax.set_title(f'{title}\nn={len(true_v)}  r={pr:+.3f}  ρ={sr:+.3f}')
    ax.set_xlabel('True normLev'); ax.set_ylabel('Predicted')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 9. How to read the result

**Step 1 (random AA in-domain) is the architecture sanity check.** Should reproduce colab10's result (Pearson ~0.85). If r < 0.7 here, something broke — go back to architecture/training before reading further panels.

**Step 2 (CATH AA distribution shift) tests whether the function generalizes from uniform to biological AA.** Real CATH has biased AA composition (some letters far more common than others) and biological sequence structure (motifs, periodicity). If r ~ same as step 1 → the encoder learned distribution-invariant features. If r drops sharply → it learned features tied to the uniform AA distribution.

**Step 3 (CATH SS cross-rep) is the headline thesis test.** SS_seq uses only 3 of the 20 alphabet letters (H, L, S). The encoder's learned features for the other 17 letters are unused; only its features for H, L, S apply. If r is still positive (≥+0.3) → the function partially transfers across the alphabet shrink. If r collapses → the encoder's features were alphabet-specific.

**Caveat about SS:** the 3-letter alphabet means SS_seq strings have very low entropy compared to AA. After perturbation, normLev distribution will skew differently (less variance, fewer high-similarity pairs achievable from few starting points). Spot-check the held-out label histogram to interpret the cross-rep r in context.

**For the thesis:** even partial cross-rep transfer is a meaningful result. The maximalist claim would be "the encoder learned an alphabet-agnostic Lev approximator." The minimalist defensible claim is "the encoder learned a function that approximates Lev on AA perturbation pairs and partially generalizes to other distributions / alphabets — the function is portable, with degradation matching the input distribution shift."

**Next step (if results are positive):** request 3Di sequences from the server, run a 4th panel testing AA→3Di cross-rep (same alphabet size, cleanest comparison). 3Di is the strongest cross-rep test for the thesis.